# 03 - Hyperparameter Optimization

Runs fixed-seed HPO using Optuna on representative assets per category.

- **Crypto** representative: BTC/USDT
- **Forex** representative: EUR/USD
- **Indices** representative: USA30IDXUSD

Each model is tuned for both horizons (h=4, h=24) using 5 Optuna trials.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

from src.utils.config import ProjectConfig
from src.experiments.hpo_runner import run_hpo, run_all_hpo
from src.models import list_models

config = ProjectConfig('..')
print(f'Available models: {list_models()}')
print(f'Categories: {config.get_categories()}')
print(f'Horizons: {config.get_horizons()}')

## Run HPO for All Models

This runs 6 models × 3 categories × 2 horizons = 36 HPO jobs, each with 5 trials.
Total: 180 training runs.

**Note:** This can take significant time. You can also run specific models:

In [ ]:
# Option 1: Run HPO for a specific model
# result = run_hpo(config, model_name='TimeMixer', category='crypto', horizon=4)
# print(f"Best params: {result['best_params']}")
# print(f"Best value: {result['best_value']:.6f}")

# Option 2: Run all HPO (uncomment to execute)
run_all_hpo(config)

# Option 3: Run for specific models only
# run_all_hpo(config, models_filter=['Autoformer', 'PatchTST'])

print('To run HPO, uncomment one of the options above.')
print('Or run from CLI: python src/main.py hpo')

## Inspect HPO Results

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Check for existing HPO results
hpo_results_dir = config.get_path('results_dir') / 'hpo'

if hpo_results_dir.exists():
    for model_dir in sorted(hpo_results_dir.iterdir()):
        if model_dir.is_dir():
            for best_file in model_dir.rglob('best_trial.json'):
                with open(best_file) as f:
                    result = json.load(f)
                print(f"\n{result['model_name']}/{result['category']}/h={result['horizon']}")
                print(f"  Best value: {result['best_value']:.6f}")
                print(f"  Best params: {result['best_params']}")
else:
    print('No HPO results yet. Run HPO first.')

In [ ]:
# View trial details for a specific model (if available)
if hpo_results_dir.exists():
    trial_files = list(hpo_results_dir.rglob('trials.csv'))
    if trial_files:
        for tf in trial_files[:3]:  # Show first 3
            print(f"\n{tf.relative_to(hpo_results_dir)}")
            df = pd.read_csv(tf)
            display(df)

print('\nHPO inspection complete. Proceed to 04_final_training.ipynb.')